In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

#Load Data
df = pd.read_csv("df_resampled.csv")

In [2]:
# Missing Value Indicators (batch creation)
# ---------------------------
missing_df = pd.DataFrame({
    f"{col}_missing": df[col].isna().astype(int)
    for col in df.columns
    if df[col].isna().sum() > 0
})



# Simple Imputation (median for numeric only)

df = df.copy()  # defragment dataframe
df.fillna(df.median(numeric_only=True), inplace=True)


# Log Transform Skewed Features (batch creation)

skewed_cols = [
    col for col in df.columns
    if df[col].dtype != 'O'           # not object
    and df[col].nunique() > 10        # continuous
    and (df[col] > 0).all()           # strictly positive
]

log_df = pd.DataFrame({
    f"{col}_log": np.log1p(df[col])
    for col in skewed_cols
})


# Interaction Features (batch creation)

interaction_pairs = [
    ("velocity_6h", "velocity_24h"),
    ("device_fraud_count", "device_distinct_emails_8w"),
    ("prev_address_months_count", "current_address_months_count"),
]

interaction_dict = {}
eps = 1e-9  # small constant to prevent divide-by-zero

for f1, f2 in interaction_pairs:
    if f1 in df.columns and f2 in df.columns:
        interaction_dict[f"{f1}_x_{f2}"] = df[f1] * df[f2]
        interaction_dict[f"{f1}_plus_{f2}"] = df[f1] + df[f2]

        denom = df[f2] + 1
        denom = np.where(denom == 0, eps, denom)  # prevent zero denominator
        interaction_dict[f"{f1}_ratio_{f2}"] = df[f1] / denom

interaction_df = pd.DataFrame(interaction_dict)


# Combine all engineered features at once

df_engineered = pd.concat([df, missing_df, log_df, interaction_df], axis=1)

df_engineered.replace([np.inf, -np.inf], np.nan, inplace=True)
df_engineered.fillna(df_engineered.median(numeric_only=True), inplace=True)

In [3]:
# Apply Scaling (StandardScaler)
# Scale ONLY feature columns (exclude target)

target_col = "fraud_bool"
feature_cols = [c for c in df_engineered.columns if c != target_col]

scaler = StandardScaler()
df_engineered[feature_cols] = scaler.fit_transform(df_engineered[feature_cols])


# Save final dataset

df_engineered.to_csv("engineered_dataset_scaled.csv", index=False)

print("Feature engineering + scaling complete. File saved as engineered_dataset_scaled.csv")

Feature engineering + scaling complete. File saved as engineered_dataset_scaled.csv


## Filter Method 1: Correlation Filter (removes redundancy)

In [4]:
def correlation_filter(X, threshold=0.95):
    corr = X.corr(numeric_only=True).abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    return to_drop

df = pd.read_csv("engineered_dataset_scaled.csv")

target = "fraud_bool"
X = df.drop(columns=[target])
y = df[target]

to_drop = correlation_filter(X, threshold=0.95)
X_corr = X.drop(columns=to_drop)

print(f"Dropped {len(to_drop)} highly correlated features.")
print("Examples:", to_drop[:10])

Dropped 8 highly correlated features.
Examples: ['income_log', 'name_email_similarity_log', 'customer_age_log', 'velocity_24h_log', 'velocity_4w_log', 'proposed_credit_limit_log', 'velocity_6h_plus_velocity_24h', 'device_fraud_count_plus_device_distinct_emails_8w']


## Feature Extraction: PCA (compresses features into components)

In [6]:
# choose components that explain 95% of variance
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_corr)

print("Original features:", X_corr.shape[1])
print("PCA components:", X_pca.shape[1])
print("Explained variance:", pca.explained_variance_ratio_.sum())

Original features: 40
PCA components: 28
Explained variance: 0.9552675609610353


In [11]:
base_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    n_jobs=-1
)

rfe = RFE(estimator=base_model, n_features_to_select=15)
rfe.fit(X_corr, y)

selected_features_rfe = X_corr.columns[rfe.support_]
X_rfe = X_corr[selected_features_rfe]

print("RFE selected features:", list(selected_features_rfe))

RFE selected features: ['income', 'name_email_similarity', 'prev_address_months_count', 'customer_age', 'days_since_request', 'date_of_birth_distinct_emails_4w', 'employment_status', 'credit_risk_score', 'housing_status', 'phone_home_valid', 'phone_mobile_valid', 'has_other_cards', 'device_os', 'keep_alive_session', 'days_since_request_log']


## Comparison (Required): Evaluate each method using a model

In [12]:
def evaluate(X, y, name):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    model = RandomForestClassifier(
        n_estimators=200, random_state=42, n_jobs=-1
    )
    model.fit(X_train, y_train)
    preds = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, preds)

    print(f"{name} -> ROC-AUC: {auc:.4f}")

# Evaluate correlation-filtered
evaluate(X_corr, y, "Correlation Filter")

# Evaluate RFE-selected
evaluate(X_rfe, y, "RFE")

# Evaluate PCA
evaluate(X_pca, y, "PCA")

Correlation Filter -> ROC-AUC: 0.9728
RFE -> ROC-AUC: 0.9653
PCA -> ROC-AUC: 0.9549
